# Beat the Floor - DCRNN

Whether **DCRNN** can be pushed past naive persistence without changing what the
network learns.

The training loop and model selection here are identical to the physics sweep.
Everything measured below is either a change to *how folds are drawn* or a
correction applied to predictions **after** training, calibrated on the
validation split and applied to test.

## Why this run exists

The headline `DCRNN`-vs-floor result is **pseudoreplicated**. Within an origin,
seed variance is 0.02-0.05; between origins it is 22.07 / 37.74 / 27.35. Nine
(origin, seed) rows are three independent observations, not nine, and clustering
by origin turns $-0.467, p=0.10$ into $-0.467, p=0.436$. This kernel runs
**9 disjoint origin blocks** instead of 3 overlapping ones.

## Arms

| arm | what it is |
|---|---|
| `raw` | the existing pipeline: $\mathrm{expm1}(m)$ |
| `smear` | Duan's smearing, $S=\overline{\exp(e)}$ on validation residuals |
| `lognorm` | the Gaussian case $S=\exp(v/2)$; heteroscedastic under a Gaussian head |
| `calib` | one factor per horizon fit against validation RMSE directly |
| `blend_*` | convex blend with persistence, weight per horizon fit on validation |
| `ens_*` | mean of the 5 seeds' count-space predictions |

`raw` is trained on `log1p` and scored on counts, so it reports a conditional
**median** where RMSE is minimised by the conditional **mean**. That predicts a
downward bias growing with residual variance -- and `error_diagnosis.json`
measures exactly that: $-12.92$ on outbreak windows against $+0.38$ on quiet
ones. The three corrections are three ways of estimating the same factor.

## 1. Environment

In [ ]:
import subprocess, sys, os, json, time, hashlib, re
from pathlib import Path

REPO = "https://github.com/MLOpenSourceOpenScience/disease_modeling_MLOS2.git"
COMMIT = "45f1c0878f002407633ed1237638734faa9ceb2b"
BLOB_NPY = "f7cfa6ec31a4058584fe256a1d6de6800e72a5b1"
BLOB_ADJ = "f3a3cb7f43998850410b0a494f16f331c3830a84"
SEGMENTS = [0.6, 0.7, 0.8, 0.9, 1.0]     # the authors' __main__ segment list

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
# Build outside /kaggle/working: anything left there becomes kernel output, and a
# 40 MB clone plus a venv makes `kaggle kernels output` unusably slow.
SCRATCH = Path("/tmp/repro") if Path("/tmp").exists() else WORK
SCRATCH.mkdir(parents=True, exist_ok=True)
SRC = SCRATCH / "mlos2"
VENV = SCRATCH / "venv311"
PY311 = VENV / "bin" / "python"

os.environ["MPLBACKEND"] = "Agg"          # their plotting helpers call plt.show()


def sh(*args, check=True, quiet=False, **kw):
    """Run a command, echoing it, and abort on a non-zero exit."""
    if not quiet:
        print("$", " ".join(str(a) for a in args))
    r = subprocess.run([str(a) for a in args], text=True, capture_output=True, **kw)
    if r.stdout.strip() and not quiet:
        print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        if check:
            raise SystemExit("command failed: " + " ".join(str(a) for a in args))
    return r


print("kernel python:", sys.version.split()[0])

In [ ]:
# Kaggle runs Python 3.12; torch 2.1.2 has no cp312 wheel. Fetch a standalone
# 3.11 with uv rather than bumping the authors' pinned torch.
sh(sys.executable, "-m", "pip", "install", "-q", "uv")

UV = [sys.executable, "-m", "uv"]
sh(*UV, "python", "install", "3.11")
sh(*UV, "venv", "--python", "3.11", str(VENV))

PIP = [*UV, "pip", "install", "-q", "--python", str(PY311)]

# Exactly the versions in the authors' requirements.txt.
sh(*PIP, "torch==2.1.2", "--index-url", "https://download.pytorch.org/whl/cpu")
sh(*PIP, "torch_scatter", "torch_sparse", "-f",
   "https://data.pyg.org/whl/torch-2.1.2+cpu.html")

# Deviation 2: the authors pin torch_geometric==2.5.3, but PGT 0.54.0 imports
# torch_geometric.utils.to_dense_adj, which PyG removed in 2.4. 2.4.0 is the
# newest release where all five architectures import.
sh(*PIP, "torch_geometric==2.4.0", "numpy~=1.26.2", "pandas~=2.2.0",
   "scikit_learn==1.4.0", "statsmodels==0.14.1", "decorator==4.4.2",
   "cython", "matplotlib", "tqdm")

# Deviation 1: PGT's own pandas<=1.3.5 pin contradicts the authors'
# pandas~=2.2.0 and has no Python 3.11 wheel.
sh(*PIP, "--no-deps", "torch_geometric_temporal==0.54.0")

In [ ]:
probe = sh(str(PY311), "-c", """
import json, torch, torch_geometric, pandas, numpy
from torch_geometric_temporal import A3TGCN, ASTGCN, AAGCN
from torch_geometric_temporal.nn.recurrent import DCRNN
from torch_geometric_temporal.signal import StaticGraphTemporalSignal, temporal_signal_split
print(json.dumps({
    "python": ".".join(map(str, __import__("sys").version_info[:3])),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "pandas": pandas.__version__,
    "numpy": numpy.__version__,
}))
""", quiet=True)

versions = json.loads(probe.stdout.strip().splitlines()[-1])
print(json.dumps(versions, indent=2))
assert versions["python"].startswith("3.11"), versions["python"]
assert versions["torch"].startswith("2.1.2"), versions["torch"]
assert versions["torch_geometric"] == "2.4.0", versions["torch_geometric"]
print("all five architectures import OK under Python 3.11")

## 2. Clone

In [ ]:
PROJECT = "https://github.com/rathishTharusha/dengue-forecasting-gnn.git"
BRANCH = "exp/beat-baseline"
PROJ = SCRATCH / "project"

if not PROJ.exists():
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, PROJECT, str(PROJ))
print("Cloned branch:", BRANCH)

RUNNER = PROJ / "analysis" / "_build" / "run_beat_baseline.py"
assert RUNNER.exists(), f"Runner not found at {RUNNER}"
ARCH = "DCRNN"

## 3. Deterministic head

Huber on `log1p` -- the existing pipeline, and the arm every correction is
measured against.

In [ ]:
started = time.time()
print("=== det head ===", flush=True)
proc = subprocess.Popen(
    [str(PY311), "-u", str(RUNNER)] + ["--arch", ARCH, "--n-origins", "9", "--seeds", "3", "--head", "det", "--out-dir", str(WORK), "--tag", ARCH],
    cwd=str(PROJ),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, f"runner failed with exit code {proc.returncode}"
print(f"\ndet head finished in {(time.time() - started) / 60:.1f} min", flush=True)

## 4. Gaussian head

The same run with a Gaussian head. This is what makes the `lognorm` correction
*heteroscedastic*: the network predicts a variance per window, so the correction
is largest on exactly the high-variance outbreak windows where the measured bias
is $-12.9$ rather than $+0.4$.

In [ ]:
started = time.time()
print("=== gauss head ===", flush=True)
proc = subprocess.Popen(
    [str(PY311), "-u", str(RUNNER)] + ["--arch", ARCH, "--n-origins", "9", "--seeds", "3", "--head", "gauss", "--out-dir", str(WORK), "--tag", ARCH],
    cwd=str(PROJ),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, f"runner failed with exit code {proc.returncode}"
print(f"\ngauss head finished in {(time.time() - started) / 60:.1f} min", flush=True)

## 5. Negative-binomial head

A count likelihood, $\mathrm{Var} = \mu + \alpha\mu^2$, emitting the conditional
**mean** directly. Where the corrections above repair a log-space median after
the fact, this never creates the problem -- there is no retransformation step to
be biased by. NB2 rather than Poisson because the target is heavily
overdispersed (median 13, max 2631, 9.7% zeros).

The `calib` factor fitted on top is a **diagnostic** here, not a repair: near
1.0 means the count likelihood really did remove the bias.

In [ ]:
started = time.time()
print("=== nb head ===", flush=True)
proc = subprocess.Popen(
    [str(PY311), "-u", str(RUNNER)] + ["--arch", ARCH, "--n-origins", "9", "--seeds", "3", "--head", "nb", "--out-dir", str(WORK), "--tag", ARCH],
    cwd=str(PROJ),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, f"runner failed with exit code {proc.returncode}"
print(f"\nnb head finished in {(time.time() - started) / 60:.1f} min", flush=True)

## 6. Every arm against the floor

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

records = []
for f in sorted(Path(WORK).glob("beat_*.json")):
    rows = json.loads(f.read_text(encoding="utf-8"))
    for r in rows:
        r["source"] = f.stem
    records.extend(rows)
df = pd.DataFrame(records)
print(f"{len(df)} records from {df['source'].nunique()} run(s)")

floor = (df[df["arch"] == "persistence"]
         .groupby("origin")["RMSE_clean"].mean().to_dict())
print("\nPersistence floor by origin (artifact-free):")
for o, v in sorted(floor.items()):
    print(f"  {o:5.2f}  {v:7.3f}")
print(f"  mean   {np.mean(list(floor.values())):7.3f}")

# Cluster by origin: average the seeds inside an origin first, so the test has
# one observation per independent fold rather than one per training run.
rows = []
model = df[df["arch"] != "persistence"]
for (src, arch, arm), grp in model.groupby(["source", "arch", "arm"]):
    per_origin = grp.groupby("origin")["RMSE_clean"].mean()
    o = sorted(per_origin.index)
    v = np.array([per_origin[k] for k in o])
    f = np.array([floor[k] for k in o])
    d = v - f
    p = stats.ttest_rel(v, f)[1] if len(v) > 1 else float("nan")
    rows.append(dict(source=src, arch=arch, arm=arm, n_origins=len(v),
                     RMSE=v.mean(), vs_floor=d.mean(),
                     wins=int((d < 0).sum()), p=p))

out = pd.DataFrame(rows).sort_values("vs_floor")
pd.set_option("display.width", 160)
print("\n=== every arm against the floor, clustered by origin ===")
print(out.to_string(index=False, float_format=lambda x: f"{x:8.4f}"))

best = out.iloc[0]
print(f"\nBest arm: {best['arch']} {best['arm']} -> {best['RMSE']:.3f} "
      f"({best['vs_floor']:+.3f} vs floor, {best['wins']}/{best['n_origins']} origins, "
      f"p={best['p']:.4f})")
if best["vs_floor"] < 0 and best["p"] < 0.05:
    print("BEATS THE FLOOR at p<0.05, clustered by origin.")
else:
    print("Does not clear the floor at p<0.05. Report as measured.")

out.to_csv(Path(WORK) / "beat_summary.csv", index=False)